In [1]:
# 1. 문서의 내용을 읽는다.
# 2. 문서를 쪼갠다
#    - 토큰 수 초과로 답변을 생성하지 못할 수 있고
#    - 문서가 길면(input이 길면) 답변 생성이 오래걸림
# 3. 임베딩  ->  벡터 데이터베이스에 저장
# 4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
# 5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [4]:
#%pip install python-dotenv langchain langchain-openai langchain-community langchain-text-splitters docx2txt langchain-chroma

In [32]:
# query -> 직장인 -> 거주자 chain 추가
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 환경변수를 불러옴
load_dotenv()

#임베딩 : 분할된 텍스트 조각들을 숫자 벡터로 변환하는 과정
# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = OpenAIEmbeddings(model='text-embedding-3-large')

index_name = 'tax-table-index'

# 데이터 추가할때
#database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

# 이미 생성된 데이터베이스를 사용할 때 
database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

llm = ChatOpenAI(model='gpt-4o')

prompt = ChatPromptTemplate.from_messages([
    ('human', '''You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:''')
])

qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()

# dictionary_chain을 qa_chain과 연결
tax_chain = {"query": dictionary_chain} | qa_chain

ai_response = tax_chain.invoke({"question": query})